# Обучение граф-гибрида (по эпохам)

Обучение гибридной граф-модели на AI2D с детальным по-эпохным просмотром метрик.

In [1]:
from pathlib import Path
import os
import sys

def _find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'vqa_retrieval').exists() and (candidate / 'experiments').exists():
            return candidate
    raise RuntimeError('Cannot find ai2d_vqa_clean root. Open the notebook from inside the clean project.')

PROJECT_ROOT = _find_project_root()
EXTERNAL_ROOT = PROJECT_ROOT.parent
AI2D_ROOT = EXTERNAL_ROOT / 'ai2d'
DOCVQA_ROOT = EXTERNAL_ROOT / 'docvqa'
INFOGRAPHICVQA_ROOT = EXTERNAL_ROOT / 'infographicvqa'
ROOT = PROJECT_ROOT
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('EXTERNAL_ROOT =', EXTERNAL_ROOT)


PROJECT_ROOT = C:\Users\Jet\Desktop\data\diagram_vqa
EXTERNAL_ROOT = C:\Users\Jet\Desktop\data


# AI2D Graph-Hybrid Training, Epoch View

Интерактивный notebook для обучения `GraphEncoderV2 + TextProj` без `argparse` и без `parser.add_argument`.

Что показывает:
- split sizes;
- stage1 / stage2 по эпохам;
- train loss, retrieval loss, VQA loss;
- validation MeanR@10, VQA accuracy, composite;
- лучший checkpoint;
- test metrics и public AI2D accuracy vs SOTA target.

In [2]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "vqa_retrieval").exists():
    ROOT = Path.cwd().parents[0]
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

from scripts.train_ai2d_hybrid import (
    build_cache_signature,
    evaluate_split,
    maybe_limit_samples,
    save_checkpoint,
    set_seed,
    train_epoch,
    write_jsonl,
)
from vqa_retrieval.ai2d_hybrid import (
    Ai2dHybridDataset,
    load_manifest_hybrid,
    load_split_payload,
    make_hybrid_collate_fn,
    select_samples_for_split,
)
from vqa_retrieval.graph_builder_v2 import FeatureCacheV2, GraphEncoderV2, NodeFeaturizerV2
from vqa_retrieval.public_vqa_metrics import evaluate_public_vqa_rows, write_public_vqa_report

print(f"ROOT={ROOT}")
print(f"CUDA={torch.cuda.is_available()} device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

ModuleNotFoundError: No module named 'torch_geometric'

## 1. Training Config

Для быстрого теста поставь `max_train_samples`, `max_val_samples`, `max_test_samples` в маленькие числа. Для полного запуска оставь `None`.

In [ ]:
CONFIG = {
    "manifest": AI2D_ROOT / "prepared_v2" / "manifest_hybrid.jsonl",
    "split_json": AI2D_ROOT / "prepared_v2" / "split_hybrid.json",
    "output_dir": ROOT / "runs" / "ai2d_hybrid_notebook_epoch_view",
    "cache_dir": AI2D_ROOT / "_cache_graph_hybrid",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
    "batch_size": 4,
    "eval_batch_size": 8,
    "num_workers": 0,
    "lr": 2e-4,
    "grad_accum_steps": 1,
    "epochs_stage1": 8,
    "epochs_stage2": 25,
    "early_stopping_patience": 6,
    "hidden_dim": 256,
    "out_dim": 256,
    "disable_attn_pool": False,
    "temperature": 0.07,
    "lambda_ret": 0.6,
    "lambda_vqa": 0.4,
    "use_caption_context": True,
    "extract_min_area": 300,
    "extract_max_nodes": 80,
    "extract_min_text_conf": 35.0,
    "extract_knn_k": 4,
    "ocr_level": "line",
    "disable_shape_nodes": False,
    "disable_cache": False,
    "disable_amp": False,
    "max_train_samples": None,
    "max_val_samples": None,
    "max_test_samples": None,
}

args = SimpleNamespace(**CONFIG)
args.output_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in CONFIG.items()}, ensure_ascii=False, indent=2))

## 2. Load Splits And DataLoaders

In [ ]:
set_seed(args.seed)
device = torch.device(args.device)

samples = load_manifest_hybrid(args.manifest)
split_payload = load_split_payload(args.split_json)

train_samples = select_samples_for_split(samples, "train", split_payload)
val_samples = select_samples_for_split(samples, "val", split_payload)
test_samples = select_samples_for_split(samples, "test", split_payload)

train_samples = maybe_limit_samples(train_samples, args.max_train_samples, seed=args.seed)
val_samples = maybe_limit_samples(val_samples, args.max_val_samples, seed=args.seed + 1)
test_samples = maybe_limit_samples(test_samples, args.max_test_samples, seed=args.seed + 2)

split_df = pd.DataFrame([
    {"split": "train", "samples": len(train_samples), "images": len({s.image_id for s in train_samples})},
    {"split": "val", "samples": len(val_samples), "images": len({s.image_id for s in val_samples})},
    {"split": "test", "samples": len(test_samples), "images": len({s.image_id for s in test_samples})},
])
display(split_df)

collate_fn = make_hybrid_collate_fn(use_caption_context=args.use_caption_context)
train_loader = DataLoader(
    Ai2dHybridDataset(train_samples),
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=args.num_workers,
    collate_fn=collate_fn,
)
eval_loader_kwargs = {
    "batch_size": args.eval_batch_size,
    "shuffle": False,
    "num_workers": args.num_workers,
    "collate_fn": collate_fn,
}
val_loader = DataLoader(Ai2dHybridDataset(val_samples), **eval_loader_kwargs)
test_loader = DataLoader(Ai2dHybridDataset(test_samples), **eval_loader_kwargs)

## 3. Build Model

Image graph encoder: `GraphEncoderV2`. Text branch: `SentenceTransformer -> TextProj MLP`.

In [ ]:
featurizer = NodeFeaturizerV2(device=str(device))
cache_signature = build_cache_signature(args, featurizer)
cache = FeatureCacheV2(
    cache_dir=args.cache_dir,
    signature=cache_signature,
    enabled=not args.disable_cache,
)

args.in_dim = featurizer.vision_dim + featurizer.text_dim + 12 + 1 + 1
gnn = GraphEncoderV2(
    in_dim=args.in_dim,
    hidden_dim=args.hidden_dim,
    out_dim=args.out_dim,
    use_attn_pool=not args.disable_attn_pool,
).to(device)
text_proj = nn.Sequential(
    nn.Linear(featurizer.text_dim, args.hidden_dim),
    nn.GELU(),
    nn.Linear(args.hidden_dim, args.out_dim),
).to(device)

optimizer = torch.optim.AdamW(list(gnn.parameters()) + list(text_proj.parameters()), lr=args.lr)
use_amp = (device.type == "cuda") and (not args.disable_amp)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print({
    "in_dim": args.in_dim,
    "vision_dim": featurizer.vision_dim,
    "text_dim": featurizer.text_dim,
    "cache_signature": cache_signature,
    "use_amp": use_amp,
})

## 4. Train Epoch By Epoch

In [3]:
history = []
best_composite = float("-inf")
best_epoch = -1
no_improve = 0
checkpoint_path = args.output_dir / "checkpoint_best.pt"
metrics_path = args.output_dir / "metrics.json"

stage_plan = [("stage1", args.epochs_stage1), ("stage2", args.epochs_stage2)]
global_epoch = 0
stop_training = False

for stage_name, stage_epochs in stage_plan:
    if stage_epochs <= 0:
        continue
    for _ in range(stage_epochs):
        global_epoch += 1
        train_stats = train_epoch(
            loader=train_loader,
            gnn=gnn,
            text_proj=text_proj,
            featurizer=featurizer,
            cache=cache,
            optimizer=optimizer,
            scaler=scaler,
            args=args,
            device=device,
            stage=stage_name,
        )
        val_result = evaluate_split(
            loader=val_loader,
            gnn=gnn,
            text_proj=text_proj,
            featurizer=featurizer,
            cache=cache,
            args=args,
            device=device,
            split_name="val",
        )
        val_metrics = val_result["metrics"]
        entry = {
            "epoch": global_epoch,
            "stage": stage_name,
            "train_loss": train_stats["loss"],
            "train_loss_ret": train_stats["loss_ret"],
            "train_loss_vqa": train_stats["loss_vqa"],
            "val_mean_r1": val_metrics["mean"]["1"],
            "val_mean_r5": val_metrics["mean"]["5"],
            "val_mean_r10": val_metrics["mean"]["10"],
            "val_vqa_acc": val_metrics["vqa_acc"],
            "val_composite": val_metrics["composite"],
        }
        history.append(entry)
        display(pd.DataFrame(history).tail(10))

        if val_metrics["composite"] > best_composite:
            best_composite = float(val_metrics["composite"])
            best_epoch = global_epoch
            no_improve = 0
            save_checkpoint(
                path=checkpoint_path,
                gnn=gnn,
                text_proj=text_proj,
                optimizer=optimizer,
                featurizer=featurizer,
                cache_signature=cache_signature,
                args=args,
                epoch_idx=global_epoch,
                stage=stage_name,
                val_metrics=val_metrics,
            )
            print(f"[BEST] epoch={global_epoch} composite={best_composite:.4f}")
        else:
            no_improve += 1
            if no_improve >= args.early_stopping_patience:
                print(f"[STOP] no improvement for {no_improve} eval steps")
                stop_training = True
                break
    if stop_training:
        break

history_df = pd.DataFrame(history)
display(history_df)
print(f"best_epoch={best_epoch} best_composite={best_composite:.4f}")

NameError: name 'args' is not defined

## 5. Plot History

In [ ]:
ax = history_df.plot(x="epoch", y=["train_loss", "train_loss_ret", "train_loss_vqa"], figsize=(10, 4), grid=True)
ax.set_title("Train losses")
ax = history_df.plot(x="epoch", y=["val_mean_r10", "val_vqa_acc", "val_composite"], figsize=(10, 4), grid=True)
ax.set_title("Validation metrics")

## 6. Test Evaluation And Public AI2D Score

In [1]:
if not checkpoint_path.exists():
    raise RuntimeError("No checkpoint_best.pt found. Run training first.")

checkpoint = torch.load(checkpoint_path, map_location=device)
gnn.load_state_dict(checkpoint["gnn_state_dict"])
text_proj.load_state_dict(checkpoint["text_proj_state_dict"])

test_result = evaluate_split(
    loader=test_loader,
    gnn=gnn,
    text_proj=text_proj,
    featurizer=featurizer,
    cache=cache,
    args=args,
    device=device,
    split_name="test",
)

write_jsonl(test_result["vqa_predictions"], args.output_dir / "test_vqa_predictions.jsonl")
write_jsonl(test_result["retrieval_predictions"], args.output_dir / "test_retrieval_predictions.jsonl")

metrics_payload = {
    "best_epoch": best_epoch,
    "best_composite": best_composite,
    "train_history": history,
    "test": test_result["metrics"],
    "config": {k: str(v) for k, v in CONFIG.items()},
}
metrics_path.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")

public_metrics, public_results = evaluate_public_vqa_rows(test_result["vqa_predictions"], dataset_name="ai2d")
public_paths = write_public_vqa_report(
    public_metrics,
    public_results,
    output_dir=args.output_dir / "public_vqa",
    prefix="ai2d_graph_hybrid",
)

display(pd.DataFrame([test_result["metrics"]]))
display(pd.DataFrame([public_metrics]))
print(f"metrics_path={metrics_path}")
print(public_paths)

NameError: name 'checkpoint_path' is not defined